# Inferência — PA2

Recebe o caminho de uma sequência do MOT17, roda o rastreador final (modelo de movimento LSTM, **sem retreinar**) e produz:

- um **vídeo** com as caixas coloridas por identidade (cor consistente ao longo do tempo);
- a **contagem de objetos únicos** (número de identidades distintas que o rastreador emitiu).

Basta ajustar `SEQ` na célula de configuração e executar tudo em ordem.

In [ ]:
import os
if not os.path.exists('/content/PA2_Deep_Learning'):
    !git clone https://github.com/RogerVin1/PA2_Deep_Learning.git
%cd /content/PA2_Deep_Learning

import glob, zipfile, urllib.request
import numpy as np
import torch, torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import defaultdict
from scipy.optimize import linear_sum_assignment
import matplotlib.animation as animation

from IPython.display import Video, display

from metrics import iou, agrupa_por_frame
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
SEQ = 'MOT17-09-SDP'      
CHECKPOINT = 'checkpoints/movimento_lstm.pt'
BAIXAR_IMAGENS = True     # False = so as caixas (sem fundo)
MAX_AGE = 5
IOU_MIN = 0.3

In [ ]:
def carregar_gt(caminho):
    arr = np.atleast_2d(np.loadtxt(caminho, delimiter=','))
    gt = []
    for r in arr:
        frame, idv, x, y, w, h, flag, classe, vis = r[:9]
        if int(flag) == 1 and int(classe) == 1:
            gt.append(dict(frame=int(frame), id=int(idv), x=float(x), y=float(y),
                           w=float(w), h=float(h), vis=float(vis)))
    return gt

def carregar_det(caminho, score_min=0.0):
    arr = np.atleast_2d(np.loadtxt(caminho, delimiter=','))
    dets = []
    for r in arr:
        frame, x, y, w, h, score = r[0], r[2], r[3], r[4], r[5], r[6]
        if float(score) >= score_min:
            dets.append(dict(frame=int(frame), x=float(x), y=float(y),
                             w=float(w), h=float(h), conf=float(score)))
    return dets

def ler_seqinfo(seqdir):
    info = {}
    with open(os.path.join(seqdir, 'seqinfo.ini')) as f:
        for linha in f:
            if '=' in linha:
                k, v = linha.strip().split('=', 1); info[k] = v
    return int(info['imWidth']), int(info['imHeight'])

def _box(d): return dict(x=d['x'], y=d['y'], w=d['w'], h=d['h'])

def box_para_estado(b, W, H):
    return np.array([(b['x']+b['w']/2)/W, (b['y']+b['h']/2)/H, b['w']/W, b['h']/H], np.float32)

def estado_para_box(s, W, H):
    cx, cy, w, h = s
    return dict(x=(cx-w/2)*W, y=(cy-h/2)*H, w=w*W, h=h*H)

In [ ]:
# modelo de movimento e rastreador 
class MovimentoRNN(nn.Module):
    CELULAS = {'rnn': nn.RNN, 'gru': nn.GRU, 'lstm': nn.LSTM}

    def __init__(self, celula='lstm', hidden=64, escala=None, dim=4):
        super().__init__()
        self.celula, self.hidden = celula, hidden
        self.rnn = self.CELULAS[celula](2 * dim, hidden, batch_first=True)
        self.cabeca = nn.Linear(hidden, dim)
        esc = torch.ones(dim) if escala is None else torch.as_tensor(escala, dtype=torch.float32)
        self.register_buffer('escala', esc.clone())

    def forward(self, x, estado=None):
        h, anterior = (None, x[:, :1]) if estado is None else estado
        delta = (x - torch.cat([anterior, x[:, :-1]], 1)) / self.escala
        saida, h = self.rnn(torch.cat([x - 0.5, delta], -1), h)
        return x + self.cabeca(saida) * self.escala, (h, x[:, -1:])

@torch.no_grad()
def rastreador_rnn(dets, modelo, W, H, iou_min=0.3, max_age=5, n_warmup=3, dev='cpu', log=None, cascata=False):
                   
    modelo.eval()
    por_frame = agrupa_por_frame(dets)
    tracks, prox_id, pred = [], 1, []
    def passo(entrada, estado):
        saida, estado = modelo(torch.tensor(entrada).view(1, 1, 4).to(dev), estado)
        return saida.view(4).cpu().numpy(), estado
    
    for t in sorted(por_frame):
        D = por_frame[t]
        for tr in tracks:
            tr['box_match'] = estado_para_box(tr['prox'], W, H) if tr['n_obs'] >= n_warmup else tr['box']
        if cascata:
            grupos = [[a for a, tr in enumerate(tracks) if tr['idade'] == 0],
                      [a for a, tr in enumerate(tracks) if tr['idade'] > 0]]
        else:
            grupos = [list(range(len(tracks)))]
        casou = {}
        for grupo in grupos:
            livres = [b for b in range(len(D)) if b not in casou.values()]
            if not grupo or not livres: continue
            M = np.array([[iou(tracks[a]['box_match'], D[b]) for b in livres] for a in grupo])
            for r, c in zip(*linear_sum_assignment(-M)):
                if M[r, c] >= iou_min:
                    casou[grupo[r]] = livres[c]
        for a, tr in enumerate(tracks):
            if a in casou:
                d = D[casou[a]]
                tr['prox'], tr['hc'] = passo(box_para_estado(d, W, H), tr['hc'])
                tr['box'] = _box(d); tr['idade'] = 0; tr['n_obs'] += 1
                pred.append(dict(frame=t, id=tr['id'], **_box(d)))
            else:
                if tr['n_obs'] >= n_warmup: tr['box'] = estado_para_box(tr['prox'], W, H)
                tr['prox'], tr['hc'] = passo(tr['prox'], tr['hc'])
                tr['idade'] += 1
        tracks = [tr for tr in tracks if tr['idade'] <= max_age]
        usados = set(casou.values())
        for c, d in enumerate(D):
            if c not in usados:
                prox, hc = passo(box_para_estado(d, W, H), None)
                tracks.append(dict(id=prox_id, box=_box(d), prox=prox, hc=hc, idade=0, n_obs=1))
                pred.append(dict(frame=t, id=prox_id, **_box(d))); prox_id += 1
    return pred

In [ ]:
ckpt = torch.load(CHECKPOINT, map_location=dev)
modelo = MovimentoRNN('lstm', hidden=64).to(dev)
modelo.load_state_dict(ckpt)
modelo.eval()
print('modelo carregado de', CHECKPOINT)
print('escala aprendida (cx,cy,w,h):', modelo.escala.cpu().numpy())

In [ ]:
if not glob.glob(f'**/{SEQ}/det/det.txt', recursive=True):
    if not os.path.exists('MOT17Labels.zip'):
        print('baixando anotacoes (~10 MB)...')
        urllib.request.urlretrieve('https://motchallenge.net/data/MOT17Labels.zip', 'MOT17Labels.zip')
    with zipfile.ZipFile('MOT17Labels.zip') as z: z.extractall()

seqdir = glob.glob(f'**/{SEQ}', recursive=True)
seqdir = [d for d in seqdir if os.path.exists(os.path.join(d, 'det', 'det.txt'))][0]
W, H = ler_seqinfo(seqdir)
dets = carregar_det(f'{seqdir}/det/det.txt', score_min=0.0)
print(f'sequencia: {seqdir}  ({W}x{H})  | {len(dets)} deteccoes')

In [ ]:
# roda o rastreador final
pred = rastreador_rnn(dets, modelo, W, H, iou_min=IOU_MIN, max_age=MAX_AGE, dev=dev)

ids_unicos = sorted(set(p['id'] for p in pred))
print(f'quadros rastreados: {len(set(p["frame"] for p in pred))}')
print(f'OBJETOS UNICOS (identidades distintas): {len(ids_unicos)}')

In [ ]:
# (opcional) baixa só as imagens desta sequencia, para o video ter fundo 
IMG_DIR = None
if BAIXAR_IMAGENS:
    base = SEQ.replace('-SDP', '')   
    achou = glob.glob(f'**/{SEQ}/img1/*.jpg', recursive=True)
    if not achou:
        print('baixando o pacote completo (~5,5 GB) e extraindo so esta sequencia...')
        if not os.path.exists('MOT17full.zip'):
            urllib.request.urlretrieve('https://motchallenge.net/data/MOT17.zip', 'MOT17full.zip')
        with zipfile.ZipFile('MOT17full.zip') as z:
            alvos = [n for n in z.namelist() if f'{SEQ}/img1/' in n and n.endswith('.jpg')]
            z.extractall(members=alvos)
        achou = glob.glob(f'**/{SEQ}/img1/*.jpg', recursive=True)
    if achou:
        IMG_DIR = os.path.dirname(sorted(achou)[0])
        print('imagens em:', IMG_DIR)
    else:
        print('nao encontrei imagens; o video sera desenhado em fundo cinza.')

In [ ]:
rng = np.random.default_rng(0)
cor_do_id = {i: rng.uniform(0.2, 1.0, 3) for i in ids_unicos}

por_frame_pred = agrupa_por_frame(pred)
frames = sorted(por_frame_pred)

def desenhar(ax, t):
    ax.clear()
    img = None
    if IMG_DIR:
        caminho = os.path.join(IMG_DIR, f'{t:06d}.jpg')
        if os.path.exists(caminho): img = plt.imread(caminho)
    ax.imshow(img if img is not None else np.full((H, W, 3), 0.15))
    for p in por_frame_pred.get(t, []):
        c = cor_do_id[p['id']]
        ax.add_patch(patches.Rectangle((p['x'], p['y']), p['w'], p['h'],
                                        fill=False, edgecolor=c, linewidth=2))
        ax.text(p['x'], p['y'] - 4, f"{p['id']}", color=c, fontsize=8, weight='bold')
    ax.set_title(f'{SEQ}  quadro {t}  |  objetos unicos ate agora: '
                 f'{len({q["id"] for f in frames if f <= t for q in por_frame_pred[f]})}', fontsize=10)
    ax.set_xlim(0, W); ax.set_ylim(H, 0); ax.axis('off')

fig, ax = plt.subplots(figsize=(W/100, H/100), dpi=100)
anim = animation.FuncAnimation(fig, lambda i: desenhar(ax, frames[i]), frames=len(frames), interval=60)
anim.save('inferencia_saida.mp4', writer='ffmpeg', dpi=100)
plt.close(fig)
print('video salvo em inferencia_saida.mp4')

In [ ]:
# resumo final e exibicao do video
print(f'RESULTADO ({SEQ})')
print(f'objetos unicos rastreados: {len(ids_unicos)}')
print(f'quadros: {len(frames)}')
display(Video('inferencia_saida.mp4', embed=True, width=640))